In [ ]:
import os
import math
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

### Sentinel-2

In [1]:
### Ref: https://sentinelhub-py.readthedocs.io/en/latest/examples/process_request.html#Example-4:-Save-downloaded-data-to-disk-and-read-it-from-disk

In [ ]:
from sentinelhub import (
    SHConfig,
    CRS,
    BBox,
    DataCollection,
    MimeType,
    MosaickingOrder,
    SentinelHubRequest,
    bbox_to_dimensions,
)

In [ ]:
config = SHConfig()
config.sh_client_id = ""
config.sh_client_secret = ""
config.sh_base_url = "https://sh.dataspace.copernicus.eu"
config.sh_token_url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"

In [ ]:
def zero_percentage(array):
    zero_count = np.count_nonzero(array == 0)
    zero_ratio = zero_count / array.size
    return zero_ratio


def download_img(
    betsiboka_coords_wgs84,
    betsiboka_bbox,
    betsiboka_size,
    resolution,
    time_interval,
    dir_name,
):
    cloud_coverage = (0, 10)
    evalscript_all_bands = """
        //VERSION=3
        
        function setup() {
            return {
                input: [{
                    bands: ["B01","B02","B03","B04","B05","B06","B07","B08","B8A","B09","B10","B11","B12"],
                    units: "DN"
                }],
                output: {
                    bands: 13,
                    sampleType: "INT16"
                }
            };
        }
    
        function evaluatePixel(sample) {
            return [sample.B01,
                    sample.B02,
                    sample.B03,
                    sample.B04,
                    sample.B05,
                    sample.B06,
                    sample.B07,
                    sample.B08,
                    sample.B8A,
                    sample.B09,
                    sample.B10,
                    sample.B11,
                    sample.B12];
        }
    """
    request_all_bands = SentinelHubRequest(
        data_folder="./all_types/" + dir_name + "/",  # Imgae Save Path
        evalscript=evalscript_all_bands,
        input_data=[
            SentinelHubRequest.input_data(
                data_collection=DataCollection.SENTINEL2_L1C.define_from(
                    "s2l1c", service_url=config.sh_base_url  # Sentinel-2 L1C dataset
                ),
                time_interval=time_interval,  # time period
                mosaicking_order=MosaickingOrder.LEAST_CC,  # least cloudy acquisitions
                # maxcc=0.1,
            )
        ],
        responses=[
            SentinelHubRequest.output_response("default", MimeType.TIFF)
        ],  # Ouput Image Format
        bbox=betsiboka_bbox,  # Interested Area
        size=betsiboka_size,  # Image Size
        config=config,
    )

    request_all_bands.custom_url_params = {
        "filter": {
            "timeRange": {"from": time_interval[0], "to": time_interval[1]},
            "cloudCoverage": cloud_coverage,
        }
    }

    # save image
    all_bands_response = request_all_bands.get_data(save_data=True)
    # print(all_bands_response)
    # print(type(all_bands_response))
    num = 0
    for single_response in all_bands_response:
        if zero_percentage(single_response[:, :, 0]) < 0.5:
            # single_response.save_data()
            num += 1
    return num

In [ ]:
# CASE 1
x, y = 53.77921, 39.45965  # Location (latitude, longitude)
start_time = "2021-01-01T00:00:00"
for i in range(30):
    betsiboka_coords_wgs84 = (x - 0.006, y - 0.0045, x + 0.006, y + 0.0045)
    resolution = 10
    cloud_coverage = (0, 10)  # Cloud coverage < 10%
    betsiboka_bbox = BBox(bbox=betsiboka_coords_wgs84, crs=CRS.WGS84)
    betsiboka_size = bbox_to_dimensions(betsiboka_bbox, resolution=resolution)
    print(f"Image shape at {resolution} m resolution: {betsiboka_size} pixels")

    time_obj = datetime.strptime(start_time, "%Y-%m-%dT%H:%M:%S")
    time_obj += timedelta(days=1)
    end_time = time_obj.strftime("%Y-%m-%dT%H:%M:%S")

    time_interval = (start_time, end_time)
    dir_name = "data/" + start_time[:10]
    os.mkdir("../../data/" + dir_name)
    num_pic = download_img(
        betsiboka_coords_wgs84,
        betsiboka_bbox,
        betsiboka_size,
        resolution,
        time_interval,
        dir_name,
    )
    print(start_time, num_pic)
    start_time = end_time

### Sentinel-2 Meta

In [ ]:
x, y = -92.23731, 19.56605
bbox = (x - 0.1, y - 0.1, x + 0.1, y + 0.1)
betsiboka_bbox = BBox(bbox=bbox, crs=CRS.WGS84)
start_time = "2023-05-17T00:00:00+00:00"
end_time = "2023-05-18T00:00:00+00:00"


time_interval = (start_time, end_time)  # Time period
data_folder = "../../data/temp"  # Saved Dir

evalscript = """
    //VERSION=3
    
    function setup() {
      return {
        input: ['B03',"sunZenithAngles", "viewZenithMean","sunAzimuthAngles","viewAzimuthMean"],
        output: [
        {
            id: "test",  // 
            bands: 1,  // Output Band Number
            sampleType: SampleType.AUTO  // Output Sample Type
        },
        {
            id: "sunZenithAngles",  // ID SOLAR_ZENITH_ANGLE
            bands: 1,  // 
            sampleType: SampleType.FLOAT32  // 
        },
        {
            id: "viewZenithMean",  // Instrument zenith angle ID
            bands: 1,  // 
            sampleType: SampleType.FLOAT32  // 
        },
        {
            id: "sunAzimuthAngles",  // Sun azimuth
            bands: 1,  // 
            sampleType: SampleType.FLOAT32  // 
        },
        {
            id: "viewAzimuthMean",  // Instrument observation azimuth angle
            bands: 1,  // 
            sampleType: SampleType.FLOAT32  // 
        }
        ]
      };
    }
    
    function evaluatePixel(sample) {
      return {
          test: [sample.B03],
          sunZenithAngles: [sample.sunZenithAngles],  //
          viewZenithMean: [sample.viewZenithMean],  // 
          sunAzimuthAngles: [sample.sunAzimuthAngles],  // 
          viewAzimuthMean: [sample.viewAzimuthMean]  // 
      };
    }
    """

# Build SentinelHub Request
request = SentinelHubRequest(
    evalscript=evalscript,
    input_data=[
        SentinelHubRequest.input_data(
            data_collection=DataCollection.SENTINEL2_L1C.define_from(
                "s2l1c", service_url=config.sh_base_url  # Sentinel-2 L1C Dataset
            ),
            time_interval=time_interval,  # Time Period
            mosaicking_order=MosaickingOrder.LEAST_CC,  # least cloudy acquisitions
            # maxcc=0.1,
        )
    ],
    responses=[
        {
            "identifier": "test",  # Solar zenith angle ID
            "format": {"type": "image/tiff"},
        },
        {
            "identifier": "sunZenithAngles",  # Solar zenith angle ID
            "format": {"type": "image/tiff"},
        },
        {
            "identifier": "viewZenithMean",  # View zenith angle ID
            "format": {"type": "image/tiff"},
        },
        {
            "identifier": "sunAzimuthAngles",  # Sun azimuth
            "format": {"type": "image/tiff"},
        },
        {
            "identifier": "viewAzimuthMean",  # View azimuth
            "format": {"type": "image/tiff"},
        },
    ],
    bbox=betsiboka_bbox,
    size=bbox_to_dimensions(betsiboka_bbox, resolution=10),
    config=config,
    data_folder="../../data/temp",
)

# Request
response = request.get_data()
sz, vz = (
    response[0]["sunZenithAngles.tif"].mean(),
    response[0]["viewZenithMean.tif"].mean(),
)
sa, va = (
    response[0]["sunAzimuthAngles.tif"].mean(),
    response[0]["viewAzimuthMean.tif"].mean(),
)
print(sz, vz, sa, va)

### EMIT

In [ ]:
import earthaccess

results = earthaccess.search_data(
    # native_id='EMIT_L1B_RAD_001_20230128T124118_2302809_033',
    short_name="EMITL1BRAD",
    # point=(87.865482,44.041006),
    temporal=("2023-10-15T14:09:21", "2023-10-15T14:09:21"),
    cloud_cover=(0, 100),
    count=100,
)
results[-1]

In [ ]:
# Download EMIT dataset
temporal_list = []  # Define
for temp in list(temporal_list):
    if temp == "2023-01-28T12:41:18":
        continue
    results = earthaccess.search_data(
        # native_id='EMIT_L1B_RAD_001_20230128T124118_2302809_033',
        short_name="EMITL1BRAD",
        # point=(-62.1123,-39.89402),
        temporal=(temp, temp),
        cloud_cover=(0, 100),
        count=100,
    )
    if len(results) == 0:
        print(temp, " is None")
        continue
    result = results[-1]
    earthaccess.download(result, "../EMIT_data/")

### ERA5

In [ ]:
import cdsapi


def download_wind(year, month, day, locations, filepath):
    c = cdsapi.Client()
    c.retrieve(
        "reanalysis-era5-single-levels",
        {
            "product_type": "reanalysis",
            "variable": [
                "10m_u_component_of_wind",
                "10m_v_component_of_wind",
            ],
            "year": year,
            "month": month,
            "day": day,
            "time": [
                "00:00",
                "01:00",
                "02:00",
                "03:00",
                "04:00",
                "05:00",
                "06:00",
                "07:00",
                "08:00",
                "09:00",
                "10:00",
                "11:00",
                "12:00",
                "13:00",
                "14:00",
                "15:00",
                "16:00",
                "17:00",
                "18:00",
                "19:00",
                "20:00",
                "21:00",
                "22:00",
                "23:00",
            ],
            "area": locations,
            "format": "netcdf",
        },
        filepath,
    )

In [ ]:
# Batch size data donwnload
p_106 = pd.read_csv("./dataset.csv")  # Define
for index, data in p_106.iterrows():
    year, month, day = data["Time"][:-1].split("-")
    x1, y1, x2, y2 = (
        math.ceil(data["plume_latitude"]),
        math.floor(data["plume_longitude"]),
        math.floor(data["plume_latitude"]),
        math.ceil(data["plume_longitude"]),
    )
    locations = [x1, y1, x2, y2]
    subname = data["subdir_name"].split("/")
    if subname[7] == "":
        filepath = "../../data/Own/" + subname[6] + "_wind/" + data["Time"] + ".nc"
    else:
        directory = "../../data/Own/" + subname[7] + "_wind/" + "/".join(subname[8:])
        filepath = directory + data["Time"] + ".nc"
        if not os.path.exists(directory):
            os.makedirs(directory)
    download_wind(year, month, day, locations, filepath)
    print(locations, filepath)